In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
from src.indicators import calculate_sma, calculate_rsi
from src.stock_screener import fetch_screener_data

# Get data & get some useful data
stock_pool = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "TSLA"]
raw_data = yf.download(stock_pool, start="2024-01-01")
close_prices = raw_data['Close']

sma_5 = calculate_sma(close_prices, window=5)
sma_20 = calculate_sma(close_prices, window=20)
rsi_14 = calculate_rsi(close_prices, window=14)

print("Features initialized")

/Users/jasonshi/Desktop/Python/finance_project/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
[*********************100%***********************]  6 of 6 completed

Features initialized


In [2]:
def create_labels(df_close, horizon=7):
    """
    Looks 'horizon' days into the future. 
    1 = price went up
    0 = price went down/flat
    """
    future_price = df_close.shift(-horizon)
    labels = (future_price > df_close).astype(int)
    
    return labels

target_labels = create_labels(close_prices, horizon=7)
print("Target labels generated")

Target labels generated


In [ ]:
ticker = "AAPL"

# Condense everything we need for ML for Apple into one dataframe
aapl_features = pd.DataFrame({
    'sma_5': sma_5[ticker],
    'sma_20': sma_20[ticker],
    'rsi': rsi_14[ticker],
    'target': target_labels[ticker]
})

aapl_clean = aapl_features.dropna()

print(f"Clean training matrix for {ticker}")
print(f"Original total days: {len(aapl_features)}")
print(f"Clean days ready for ML: {len(aapl_clean)}")
print("\nFirst 5 rows of training data:")
print(aapl_clean.head())

Clean training matrix for AAPL
Original total days: 603
Clean days ready for ML: 584

First 5 rows of training data:
                 sma_5      sma_20        rsi  target
Date                                                 
2024-01-30  190.021075  185.787496  39.183851       1
2024-01-31  188.023682  185.726189  33.682860       1
2024-02-01  186.578046  185.855229  39.830815       1
2024-02-02  185.278757  186.050025  38.262465       0
2024-02-05  184.477820  186.371388  42.667382       0


In [ ]:
all_stock_matrices = []

# Put data for all stocks in dataframe
for ticker in stock_pool:
    stock_df = pd.DataFrame({
        'sma_5': sma_5[ticker],
        'sma_20': sma_20[ticker],
        'rsi': rsi_14[ticker],
        'target': target_labels[ticker]
    })
    
    stock_df['ticker'] = ticker
    all_stock_matrices.append(stock_df)

master_df = pd.concat(all_stock_matrices)
master_df_clean = master_df.dropna()

print("Global training matrix")
print(f"Total rows across all stocks combined: {len(master_df_clean)}")
print("\nRandom sample of 10 rows from the master dataset:")
print(master_df_clean.sample(10))

Global training matrix
Total rows across all stocks combined: 3504

Random sample of 10 rows from the master dataset:
                 sma_5      sma_20        rsi  target ticker
Date                                                        
2026-04-23  201.151999  186.531500  64.645031       0   NVDA
2026-01-29  432.635999  437.965001  38.771106       1   TSLA
2024-05-16  185.655768  175.464988  71.383373       1   AAPL
2025-11-25  304.374493  287.745590  76.782389       0  GOOGL
2024-11-18  144.632626  142.001684  50.904336       0   NVDA
2024-01-31  188.023682  185.726189  33.682860       1   AAPL
2025-03-10  233.476361  238.201895  37.744889       0   AAPL
2025-06-04  206.059998  203.567500  59.836520       1   AMZN
2025-10-30  529.548230  518.437100  54.186521       0   MSFT
2024-10-11  184.832001  187.401001  56.424040       1   AMZN


In [5]:
"""
X = features (training data)
y = answer key (whether or not it actually went up)
"""
X = master_df_clean.drop(columns=['target', 'ticker'])
y = master_df_clean['target']

split_index = int(len(master_df_clean) * 0.8)

# Note: We don't use train_test_split() here because stock prices depend on previous ones
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print("Data split info:")
print(f"Training set size (X_train): {X_train.shape[0]} days of data")
print(f"Testing set size (X_test):   {X_test.shape[0]} days of data")
print(f"Features the model will see: {list(X.columns)}")

Data split info:
Training set size (X_train): 2803 days of data
Testing set size (X_test):   701 days of data
Features the model will see: ['sma_5', 'sma_20', 'rsi']


In [6]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_svm = LinearSVC(max_iter=10000, random_state=42)
model_svm.fit(X_train_scaled, y_train)

predictions = model_svm.predict(X_test_scaled)
accuracy = accuracy_score(y_test, predictions)

print("LinearSVC Training:")
print(f"Model accuracy on unseen data: {accuracy * 100:.2f}%")

LinearSVC Training:
Model accuracy on unseen data: 51.36%


/Users/jasonshi/Desktop/Python/finance_project/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/jasonshi/Desktop/Python/finance_project/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/jasonshi/Desktop/Python/finance_project/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# n_estimators=100 is an assembly of 100 decision trees
model_rf = RandomForestClassifier(n_estimators=100, random_state=42, min_samples_split=10)

# Random Forest doesn't require scaling
model_rf.fit(X_train, y_train)
rf_predictions = model_rf.predict(X_test)

rf_accuracy = accuracy_score(y_test, rf_predictions)

print("Random Forest Results:")
print(f"Model accuracy on unseen data: {rf_accuracy * 100:.2f}%\n")

print(classification_report(y_test, rf_predictions))

Random Forest Results:
Model accuracy on unseen data: 53.07%

              precision    recall  f1-score   support

           0       0.52      0.43      0.47       343
           1       0.53      0.63      0.58       358

    accuracy                           0.53       701
   macro avg       0.53      0.53      0.53       701
weighted avg       0.53      0.53      0.53       701

